<a href="https://colab.research.google.com/github/DaniilDonskoy/building_maintenance_agents/blob/feature%2Fincident-data-analysis/%D0%98%D0%BD%D1%86%D0%B8%D0%B4%D0%B5%D0%BD%D1%82%D1%8B_%D1%81_%D1%82%D1%80%D1%83%D0%B1%D0%B0%D0%BC%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install natasha -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 100.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd

from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc
import re

from types import SimpleNamespace

# Загрузка таблицы

In [ ]:
sheet_id = "1h0Gt9fzYZX27zZBAdy-zkCupstS2_Qse"
sheet_name = "Sheet1"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)
df.head()

,п/п,Адрес МКД,Кол-во протечек за 2023-2024 (ХВС; ГВС ),Мероприятия,Выполнено за 2023-2024г.г.,Запланировано,Год ввода дома в эксплуатацию,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,1,"ул. Михаила Дудина, д. 25, к. 2, лит. А",137,частичная замена трубопровода ХВС (стояки и ц...,"5357,3",3800.0,2011,NaN,NaN,NaN,NaN,NaN
1,2,"ул. Михаила Дудина, д. 23, к. 1, лит. А",41,замена трубопровода ГВС (стояки и циркуляция )...,"10346,8",4540.0,2011,NaN,NaN,NaN,NaN,NaN
2,3,"ул. Федора Абрамова, д. 4, лит. А",181,частичная замена трубопровода ХВС (стояки и ц...,"3682,8",1950.0,2011,NaN,NaN,NaN,NaN,NaN
3,4,"ул. Михаила Дудина, д. 25, к. 1, лит. А",82,частичная замена трубопровода ХВС (стояки и ц...,"1532,8",870.0,2011,441.0,NaN,2011.0,441.0,"110,25"
4,5,"ул. Николая Рубцова, д. 12, к. 1, лит. А",93,частичная замена трубопровода ГВС (стояки и ц...,"3674,6",NaN,2012,NaN,NaN,2012.0,230.0,46


In [ ]:
print(df.columns)

cols2drop = ['Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
print(cols2drop)

Index(['п/п ', 'Адрес МКД', 'Кол-во протечек за 2023-2024 (ХВС; ГВС )',
       'Мероприятия ', 'Выполнено за 2023-2024г.г.', 'Запланировано',
       'Год ввода дома в эксплуатацию', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11'],
      dtype='object')
['Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']


In [ ]:
df.drop(columns=cols2drop, inplace=True, errors='ignore')

In [ ]:
df['Мероприятия '].value_counts()

,count
Мероприятия,
проводятся анализ состояния внутрених стенок трубопровода,20
проводятся анализ состояния внутрених стенок трубопровода + устранение течи радиатора,17
"частичная замена трубопровода ХВС (стояки и циркуляция ) + устранение течи радиатора, стояка",3
"проводятся анализ состояния внутрених стенок трубопровода + устранение течи радиатора, стояка",3
"замена трубопровода ГВС (стояки и циркуляция ) + устранение течи радиатора, стояка",1
проводятся анализ состояния внутрених стенок трубопровода + низкие параметры с ГУП ТЭК,1
"частичная замена трубопровода ГВС (стояки и циркуляция ) + устранение течи радиатора, стояка",1
"заменен участок трубопроввода ХВС в уровне цокольного этажа + устранение течи радиатора, стояка",1
"заменены глваные стояки ГВС + устранение течи радиатора, стояка",1


Предобработка текста

In [ ]:
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

def lemmatize_natasha(text):
    if pd.isna(text):
        return ''

    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    lemmas = []
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        lemmas.append(token.lemma)

    return ' '.join(lemmas)

In [ ]:
df['clean'] = df['Мероприятия '].apply(lemmatize_natasha)
df['clean'].value_counts()

,count
clean,
проводиться анализ состояние внутрений стенка трубопровод,20
проводиться анализ состояние внутрений стенка трубопровод устранение теча радиатор,17
проводиться анализ состояние внутрений стенка трубопровод устранение теча радиатор стояк,3
частичный замена трубопровод хвс стояк и циркуляция устранение теча радиатор стояк,3
запланировать замена главный стояк гвс устранение теча радиатор стояк,2
замена трубопровод гвс стояк и циркуляция устранение теча радиатор стояк,1
частичный замена трубопровод гвс стояк и циркуляция устранение теча радиатор стояк,1
проводиться анализ состояние внутрений стенка трубопровод низкий параметр с гуп тэк,1
замена главный стояк гвс устранение теча радиатор стояк,1


In [ ]:
def count_patterns(col, patterns):
    return {
        name: col.str.contains(pattern, na=False).sum()
        for name, pattern in patterns.items()
    }

work_patterns = {
    'analysis': r'анализ|обследован',
    'repair': r'устранить|ремонт',
    'replace': r'замен|демонтаж|монтаж',
    'planned': r'запланировать',
}

incident_patterns = {
    'leak': r'теча|протечка',
    'low_pressure': r'низкий параметр|давление',
}

system_patterns = {
    'gvs': r'гвс|горячий водоснабжение',
    'hvs': r'хвс|холодный водоснабжение',
    'radiator': r'радиатор',
    'stoyak': r'стояк',
    'pipeline': r'трубопровод',
}

In [ ]:
col = df['clean']

work_stat = count_patterns(df['clean'], work_patterns)
incident_stat = count_patterns(df['clean'], incident_patterns)
system_stat = count_patterns(df['clean'], system_patterns)

display(work_stat)
print()

display(incident_stat)
print()

display(system_stat)

{'analysis': np.int64(41),
 'repair': np.int64(0),
 'replace': np.int64(19),
 'planned': np.int64(4)}

{'leak': np.int64(39), 'low_pressure': np.int64(1)}

{'gvs': np.int64(12),
 'hvs': np.int64(5),
 'radiator': np.int64(39),
 'stoyak': np.int64(20),
 'pipeline': np.int64(49)}

# Выделенные инциденты

In [ ]:
col = df['clean']

stat = SimpleNamespace()

# 1. Анализ труб (именно трубопровод)
stat.analysis_pipe = col.str.contains(
    r'анализ.*трубопровод',
    na=False
).sum()

# 2. Утечка радиатора
stat.leak_radiator = col.str.contains(
    r'теча.*радиатор',
    na=False
).sum()

# 3. Утечка стояка
stat.leak_stoyak = col.str.contains(
    r'теча.*стояк',
    na=False
).sum()

# 4. Замена трубы ГВС
stat.replace_gvs = col.str.contains(
    r'замен.*трубопровод.*гвс',
    na=False
).sum()

# 5. Замена трубы ХВС
stat.replace_hvs = col.str.contains(
    r'замен.*трубопровод.*хвс',
    na=False
).sum()

# 6. Замена ХВС на уровне тех этажа
stat.replace_hvs_tech = col.str.contains(
    r'замен.*трубопровод.*хвс.*технический этаж',
    na=False
).sum()

# 7. Замена главных стояков ГВС (включая запланированные)
stat.replace_main_stoyak_gvs = col.str.contains(
    r'(замен|запланировать).*(главный стояк).*гвс',
    na=False
).sum()

# 8. Замена розлива
stat.replace_rozliv = col.str.contains(
    r'замен.*розлив',
    na=False
).sum()

/tmp/ipykernel_188/1504727227.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stat.replace_main_stoyak_gvs = col.str.contains(


In [ ]:
stat_df = pd.DataFrame([
    ('Анализ труб', stat.analysis_pipe),
    ('Утечка радиатора', stat.leak_radiator),
    ('Утечка стояка', stat.leak_stoyak),
    ('Замена трубы ГВС', stat.replace_gvs),
    ('Замена трубы ХВС', stat.replace_hvs),
    ('Замена ХВС на уровне тех этажа', stat.replace_hvs_tech),
    ('Замена главных стояков ГВС', stat.replace_main_stoyak_gvs),
    ('Замена розлива', stat.replace_rozliv),
], columns=['incident', 'count'])

In [ ]:
stat_df

,incident,count
0,Анализ труб (именно трубопровод),41
1,Утечка радиатора,39
2,Утечка стояка,16
3,Замена трубы ГВС,4
4,Замена трубы ХВС,4
5,Замена ХВС на уровне тех этажа,1
6,Замена главных стояков ГВС,6
7,Замена розлива,5
